# Original vs. Sandboxed Agent Accuracy

Compare Copilot-agent accuracy for the original and sandboxed CABRA runs.

- Both datasets are restricted to the four agents present in both result sets.
- Original curves average every available original score record at each experiment size to reduce variance.
- Sandboxed curves average every completed sandbox score record.

In [ ]:
from __future__ import annotations

import gzip
import json
import re
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from tqdm.auto import tqdm

DATA_ROOT = Path("../local_data")
ORIGINAL_RESULTS_ROOT = DATA_ROOT / "results"
SANDBOX_RESULTS_ROOT = DATA_ROOT / "results_sandboxed"

AGENTS = [
    "copilot/claude-opus-4.7",
    "copilot/gpt-5.5",
    "copilot/gpt-5.4-mini",
    "copilot/gemini-3.5-flash",
]
TASK_TYPES = [
    "dead_code",
    "add_parameter",
    "add_return_value",
    "cache_function",
    "extract_helper",
]
TASK_ORDER = [
    "Function Traversal",
    "Function Search",
    "Instruction Following",
    "Runtime Resolution",
    "Merge Codebases",
]
EXPERIMENTS = {
    "function_traversal": {
        "name": "Function Traversal",
        "n_pattern": re.compile(r"_en=\((?P<lo>[^,]+),(?P<hi>[^)]+)\)"),
        "task_set": "math_function",
        "score_kinds": ("behavioral",),
        "dead_code_kinds": ("behavioral", "function_existence"),
    },
    "function_search": {
        "name": "Function Search",
        "n_pattern": re.compile(r"_cn=\((?P<lo>[^,]+),(?P<hi>[^)]+)\)"),
        "task_set": "math_function",
        "score_kinds": ("behavioral",),
        "dead_code_kinds": ("behavioral", "function_existence"),
    },
    "add_instructions": {
        "name": "Instruction Following",
        "n_pattern": re.compile(r"_nc=\((?P<lo>[^,]+),(?P<hi>[^)]+)\)"),
        "task_set": "math_mult_constraints",
        "score_kinds": ("behavioral", "constraint_structure"),
        "dead_code_kinds": ("behavioral", "function_existence"),
    },
    "runtime_resolution": {
        "name": "Runtime Resolution",
        "n_pattern": re.compile(r"_ifhops=(?P<value>[^_]+)"),
        "task_set": "math_function",
        "score_kinds": ("behavioral", "branch_placement"),
        "dead_code_kinds": ("behavioral", "branch_placement", "function_existence"),
    },
}

TASK_X_LABELS = {
    "Function Traversal": "Number of Edit Functions (N)",
    "Function Search": "Number of Non-Edit Functions (N)",
    "Instruction Following": "Number of Instructions (N)",
    "Runtime Resolution": "Number of Runtime Steps (N)",
    "Merge Codebases": "Total Lines of Code (N)",
}

In [ ]:
MERGE_N_PATTERN = re.compile(r"_n=(?P<value>[^_]+)")


def open_jsonl(path: Path):
    if path.suffix == ".gz":
        return gzip.open(path, "rt", encoding="utf-8")
    return path.open(encoding="utf-8")


def experiment_key_from_run_name(run_name: str) -> str | None:
    return next((key for key in EXPERIMENTS if run_name.startswith(f"{key}_")), None)


def n_value_from_run_name(run_name: str, experiment_key: str) -> float | None:
    match = EXPERIMENTS[experiment_key]["n_pattern"].search(run_name)
    if match is None:
        return None
    if "value" in match.groupdict():
        return float(match.group("value"))
    return (float(match.group("lo")) + float(match.group("hi"))) / 2


def kind_passes(summary: dict, kind: str) -> bool:
    counts = summary.get(kind)
    return bool(
        counts
        and counts.get("pass", 0) > 0
        and counts.get("fail", 0) == 0
        and counts.get("error", 0) == 0
    )


def record_passes(record: dict, experiment_key: str, task_type: str) -> bool:
    if record.get("skipped"):
        return False
    spec = EXPERIMENTS[experiment_key]
    kinds = spec["dead_code_kinds"] if task_type == "dead_code" else spec["score_kinds"]
    summary = record.get("summary") or {}
    return all(kind_passes(summary, kind) for kind in kinds)


def collect_agent_scores(results_root: Path, source: str) -> pd.DataFrame:
    if not results_root.exists():
        raise FileNotFoundError(f"results directory not found: {results_root}")

    rows = []
    paths = sorted([*results_root.glob("**/*.jsonl"), *results_root.glob("**/*.jsonl.gz")])
    for path in tqdm(paths, desc=f"Loading {source}"):
        parts = path.relative_to(results_root).parts
        if len(parts) < 5:
            continue

        run_name, task_set_name, task_type = parts[:3]
        experiment_key = experiment_key_from_run_name(run_name)
        if experiment_key is None or task_type not in TASK_TYPES:
            continue
        if task_set_name != EXPERIMENTS[experiment_key]["task_set"]:
            continue

        model = "/".join(parts[3:]).removesuffix(".gz").removesuffix(".jsonl")
        if model not in AGENTS:
            continue

        n_value = n_value_from_run_name(run_name, experiment_key)
        if n_value is None:
            continue

        with open_jsonl(path) as file:
            for record_index, line in enumerate(file):
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                task_id = str(record.get("dag_id") or record.get("name") or "")
                if not task_id:
                    raise ValueError(f"record has no dag_id or name: {path}:{record_index + 1}")
                rows.append(
                    {
                        "source": source,
                        "experiment_key": experiment_key,
                        "task_name": EXPERIMENTS[experiment_key]["name"],
                        "N": n_value,
                        "run_name": run_name,
                        "task_set_name": task_set_name,
                        "task_type": task_type,
                        "model": model,
                        "task_id": task_id,
                        "record_index": record_index,
                        "score": int(record_passes(record, experiment_key, task_type)),
                    }
                )

    if not rows:
        raise ValueError(f"no matching agent score rows found under {results_root}")
    return pd.DataFrame(rows)


def collect_merge_scores(results_root: Path, source: str) -> pd.DataFrame:
    rows = []
    paths = sorted({
        *results_root.glob("merge_codebases_*/copilot/*.jsonl"),
        *results_root.glob("merge_codebases_*/copilot/*.jsonl.gz"),
    })
    for path in tqdm(paths, desc=f"Loading {source} merge scores"):
        parts = path.relative_to(results_root).parts
        run_name = parts[0]
        match = MERGE_N_PATTERN.search(run_name)
        if match is None:
            continue
        model = f"copilot/{parts[-1].removesuffix('.gz').removesuffix('.jsonl')}"
        if model not in AGENTS:
            continue

        with open_jsonl(path) as file:
            for record_index, line in enumerate(file):
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                summary = record.get("summary") or {}
                rows.append(
                    {
                        "source": source,
                        "experiment_key": "merge_codebases",
                        "task_name": "Merge Codebases",
                        "N": float(match.group("value")),
                        "run_name": run_name,
                        "task_set_name": None,
                        "task_type": "merge_codebases",
                        "model": model,
                        "task_id": str(record.get("dag_id") or record.get("name") or record_index),
                        "record_index": record_index,
                        "score": int(
                            not record.get("skipped")
                            and kind_passes(summary, "behavioral")
                            and kind_passes(summary, "difference_blocks")
                        ),
                    }
                )

    if not rows:
        raise ValueError(f"no matching merge score rows found under {results_root}")
    return pd.DataFrame(rows)


original_all = collect_agent_scores(ORIGINAL_RESULTS_ROOT, "Original")
sandbox_all = collect_agent_scores(SANDBOX_RESULTS_ROOT, "Sandboxed")
original_merge_all = collect_merge_scores(ORIGINAL_RESULTS_ROOT, "Original")
sandbox_merge_all = collect_merge_scores(SANDBOX_RESULTS_ROOT, "Sandboxed")

original_all.groupby(["task_name", "model"]).size().unstack(fill_value=0)

In [ ]:
comparison_records = pd.concat(
    [
        original_all,
        original_merge_all,
        sandbox_all,
        sandbox_merge_all,
    ],
    ignore_index=True,
)

coverage = (
    comparison_records.groupby(["source", "task_name", "model"], as_index=False)
    .agg(records=("score", "size"), task_ids=("task_id", "nunique"))
    .sort_values(["source", "task_name", "model"])
)
coverage

In [ ]:
from matplotlib.ticker import NullFormatter

MODEL_LABELS = {
    "copilot/claude-opus-4.7": "Opus 4.7",
    "copilot/gpt-5.5": "GPT-5.5",
    "copilot/gpt-5.4-mini": "GPT-5.4 Mini",
    "copilot/gemini-3.5-flash": "Gemini 3.5 Flash",
}
PRIORITY_MODEL_COLORS = {
    "copilot/gpt-5.5": "tab:blue",
    "copilot/gpt-5.4-mini": "tab:red",
}
LEGEND_AGENTS = [
    "copilot/gpt-5.4-mini",
    "copilot/gpt-5.5",
    "copilot/claude-opus-4.7",
    "copilot/gemini-3.5-flash",
]
ACCURACY_PLOTS_DIR = Path("sandbox-comparison-plots")
ACCURACY_PLOTS_DIR.mkdir(exist_ok=True)


def aggregate_scores(data: pd.DataFrame) -> pd.DataFrame:
    summary = (
        data.groupby(["source", "task_name", "N", "model"], as_index=False)
        .agg(passes=("score", "sum"), total=("score", "size"))
    )
    summary["score"] = summary["passes"] / summary["total"]
    summary["stderr"] = np.sqrt(summary["score"] * (1 - summary["score"]) / summary["total"])
    return summary


def model_colors() -> dict[str, tuple]:
    reserved = {mcolors.to_rgb(color) for color in PRIORITY_MODEL_COLORS.values()}
    cmap = plt.get_cmap("tab10")
    palette = [cmap(index) for index in range(cmap.N) if cmap(index)[:3] not in reserved]
    colors = {}
    palette_index = 0
    for model in AGENTS:
        if model in PRIORITY_MODEL_COLORS:
            colors[model] = mcolors.to_rgba(PRIORITY_MODEL_COLORS[model])
        else:
            colors[model] = palette[palette_index]
            palette_index += 1
    return colors


def format_axis_value(value: float) -> str:
    return str(int(value)) if float(value).is_integer() else f"{value:g}"


def make_sandbox_comparison_plot(plot_data: pd.DataFrame) -> None:
    sources = ["Original", "Sandboxed"]
    colors = model_colors()
    fig, axes = plt.subplots(
        len(sources),
        len(TASK_ORDER),
        figsize=(2.4 * len(TASK_ORDER), 4),
        squeeze=False,
    )

    for row, source in enumerate(sources):
        for col, task_name in enumerate(TASK_ORDER):
            ax = axes[row][col]
            panel = plot_data[
                (plot_data["source"] == source)
                & (plot_data["task_name"] == task_name)
            ]

            if panel.empty:
                ax.text(0.5, 0.5, "No results", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue

            for model in AGENTS:
                line = panel[panel["model"] == model].sort_values("N")
                if line.empty:
                    continue
                ax.errorbar(
                    line["N"],
                    line["score"],
                    yerr=line["stderr"],
                    marker="o",
                    markersize=3.5,
                    linewidth=1.8,
                    capsize=2.5,
                    color=colors[model],
                    label=MODEL_LABELS[model],
                )

            x_values = sorted(panel["N"].unique())
            ax.set_xscale("log")
            ax.set_xlim(min(x_values) / 1.08, max(x_values) * 1.08)
            ax.set_ylim(-0.03, 1.03)
            ax.set_xticks(x_values)
            ax.set_xticklabels([format_axis_value(value) for value in x_values])
            ax.get_xaxis().set_minor_formatter(NullFormatter())
            ax.grid(True, alpha=0.25)

            if row == 0:
                ax.set_title(task_name, fontsize=13)
                ax.tick_params(labelbottom=False)
            else:
                ax.set_xlabel(TASK_X_LABELS[task_name], fontsize=9)
            if col == 0:
                ax.set_ylabel(f"{source} Agent Acc.")

    handles = [
        Line2D([0], [0], color=colors[model], marker="o", markersize=5, linewidth=1.8)
        for model in LEGEND_AGENTS
    ]
    legend = fig.legend(
        handles,
        [MODEL_LABELS[model] for model in LEGEND_AGENTS],
        title="Agents:",
        loc="center left",
        bbox_to_anchor=(0.96, 0.5),
        fontsize=8,
        title_fontsize=9,
        frameon=True,
    )
    legend.set_alignment("left")
    fig.tight_layout(rect=(0, 0, 0.95, 1), h_pad=1)
    fig.savefig(ACCURACY_PLOTS_DIR / "agent-accuracy-sandboxed.pdf", bbox_inches="tight")
    plt.show()


comparison_plot_df = aggregate_scores(comparison_records)
make_sandbox_comparison_plot(comparison_plot_df)

## Tool Call Comparison

In [ ]:
MATH_TASK_ORDER = [
    "Function Traversal",
    "Function Search",
    "Instruction Following",
    "Runtime Resolution",
]
ORIGINAL_TOOL_CALL_ROOT = DATA_ROOT / "tool_call_results"
SANDBOX_TOOL_CALL_LABELS = DATA_ROOT / "tool_call_results_sandboxed" / "tool-call-labels-sandboxed.jsonl"

TOOL_GROUPS = [
    ("understand", ("understand",), "Analyze"),
    ("read", ("read",), "Read"),
    ("search", ("search",), "Search"),
    ("edit", ("edit",), "Edit"),
    ("test", ("test",), "Test"),
    ("other", ("summary", "plan", "file", "other"), "Other"),
]
TOOL_LABEL_TO_GROUP = {
    label: (group_key, display)
    for group_key, labels, display in TOOL_GROUPS
    for label in labels
}


def collect_tool_call_records(
    label_paths: list[Path],
    source: str,
) -> pd.DataFrame:
    rows = []
    for path in tqdm(label_paths, desc=f"Loading {source} tool labels"):
        with open_jsonl(path) as file:
            for line in file:
                if not line.strip():
                    continue
                record = json.loads(line)
                record_run_name = str(record.get("run_name") or record.get("experiment") or "")
                filename_run_name = path.name.split("__", 1)[0]
                run_name = (
                    record_run_name
                    if experiment_key_from_run_name(record_run_name) is not None
                    else filename_run_name
                )
                experiment_key = experiment_key_from_run_name(run_name)
                if experiment_key is None:
                    continue

                spec = EXPERIMENTS[experiment_key]
                task_set_name = str(record.get("task_set_name") or "")
                task_type = str(record.get("task_type") or "")
                model = str(record.get("model") or "")
                dag_id = str(record.get("dag_id") or "")
                session_id = str(record.get("session_id") or "")
                execution_id = session_id or "|".join(
                    (run_name, task_set_name, task_type, model, dag_id)
                )
                tool_call_index = record.get("tool_call_index")
                tool_group = TOOL_LABEL_TO_GROUP.get(record.get("label"))
                if (
                    task_set_name != spec["task_set"]
                    or model not in AGENTS
                    or not dag_id and not session_id
                    or not isinstance(tool_call_index, int)
                    or tool_group is None
                ):
                    continue

                n_value = n_value_from_run_name(run_name, experiment_key)
                if n_value is None:
                    continue
                rows.append(
                    {
                        "source": source,
                        "session_id": execution_id,
                        "task_name": spec["name"],
                        "N": n_value,
                        "model": model,
                        "tool_key": tool_group[0],
                        "tool_display": tool_group[1],
                    }
                )

    if not rows:
        raise ValueError(f"no matching {source} math tool calls found")
    return pd.DataFrame(rows)


def build_tool_call_comparison(calls: pd.DataFrame) -> pd.DataFrame:
    session_columns = ["source", "session_id", "task_name", "N", "model"]
    session_meta = calls[session_columns].drop_duplicates()
    groups = pd.DataFrame(
        [(key, display) for key, _labels, display in TOOL_GROUPS],
        columns=["tool_key", "tool_display"],
    )
    session_grid = session_meta.merge(groups, how="cross")
    observed = (
        calls.groupby(session_columns + ["tool_key", "tool_display"], as_index=False)
        .agg(num_tool_calls=("tool_key", "size"))
    )
    comparison = session_grid.merge(
        observed,
        on=session_columns + ["tool_key", "tool_display"],
        how="left",
    )
    comparison["num_tool_calls"] = comparison["num_tool_calls"].fillna(0)
    return comparison


original_tool_paths = sorted({
    *ORIGINAL_TOOL_CALL_ROOT.glob("*.jsonl"),
    *ORIGINAL_TOOL_CALL_ROOT.glob("*.jsonl.gz"),
})
original_tool_calls = collect_tool_call_records(original_tool_paths, "Original")
sandbox_tool_calls = collect_tool_call_records([SANDBOX_TOOL_CALL_LABELS], "Sandboxed")
tool_call_comparison_df = build_tool_call_comparison(
    pd.concat([original_tool_calls, sandbox_tool_calls], ignore_index=True)
)

math_tool_coverage = (
    tool_call_comparison_df[["source", "session_id", "task_name", "model"]]
    .drop_duplicates()
    .groupby(["source", "task_name", "model"], as_index=False)
    .size()
    .rename(columns={"size": "sessions"})
)
math_tool_coverage

In [ ]:
TOOL_COMPARISON_PLOTS_DIR = Path("sandbox-comparison-plots")
TOOL_COMPARISON_PLOTS_DIR.mkdir(exist_ok=True)
COMPARISON_SOURCES = ["Original", "Sandboxed"]


def summarize_tool_calls(data: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    return (
        data.groupby(group_columns, as_index=False)["num_tool_calls"]
        .agg(["mean", "sem"])
        .reset_index()
        .rename(columns={"sem": "stderr"})
    )


def style_tool_comparison_axis(ax, x_values: list[float], x_label: str | None = None) -> None:
    positive_x_values = sorted(value for value in x_values if value > 0)
    if not positive_x_values:
        raise ValueError("log-scaled tool-call plots require positive N values")
    ax.set_xscale("log")
    ax.set_xlim(positive_x_values[0] / 1.08, positive_x_values[-1] * 1.08)
    ax.set_xticks(positive_x_values)
    ax.set_xticklabels([format_axis_value(value) for value in positive_x_values])
    ax.get_xaxis().set_minor_formatter(NullFormatter())
    if x_label is not None:
        ax.set_xlabel(x_label, fontsize=9)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.25)


def make_tool_call_plot_by_tool_column_comparison(
    tool_plot_df: pd.DataFrame,
    task_names: list[str],
    tool_names: list[str],
) -> None:
    plot_data = tool_plot_df[
        tool_plot_df["task_name"].isin(task_names)
        & tool_plot_df["tool_display"].isin(tool_names)
    ]
    task_panels = [task for task in task_names if task in set(plot_data["task_name"])]
    active_tools = [tool for tool in tool_names if tool in set(plot_data["tool_display"])]
    if not task_panels or not active_tools:
        raise ValueError("no matching math task/tool records to plot")

    summary = summarize_tool_calls(
        plot_data,
        ["source", "task_name", "N", "tool_key", "tool_display"],
    )
    task_colors = {
        task: plt.get_cmap("tab10")(index)
        for index, task in enumerate(task_panels)
    }
    fig, axes = plt.subplots(
        len(COMPARISON_SOURCES),
        len(active_tools),
        figsize=(2.2 * len(active_tools), 4.3),
        squeeze=False,
        sharex="col",
    )
    for row, source in enumerate(COMPARISON_SOURCES):
        for col, tool_name in enumerate(active_tools):
            ax = axes[row][col]
            column_data = summary[summary["tool_display"] == tool_name]
            panel = column_data[column_data["source"] == source]
            for task_name in task_panels:
                line = panel[panel["task_name"] == task_name].sort_values("N")
                if line.empty or line["mean"].isna().all():
                    continue
                ax.errorbar(
                    line["N"],
                    line["mean"],
                    yerr=line["stderr"],
                    color=task_colors[task_name],
                    marker="o",
                    linewidth=1.6,
                    markersize=3.5,
                    capsize=2.0,
                )

            if row == 0:
                ax.set_title(tool_name, fontsize=12)
                ax.tick_params(labelbottom=False)
            if col == 0:
                ax.set_ylabel(f"{source}\nMean Tool Calls")
            style_tool_comparison_axis(
                ax,
                sorted(column_data["N"].unique()),
                "Task Complexity (N)" if row == len(COMPARISON_SOURCES) - 1 else None,
            )

    task_handles = [
        Line2D([0], [0], color=task_colors[task], marker="o", linewidth=1.8)
        for task in task_panels
    ]
    fig.legend(
        task_handles,
        task_panels,
        title="Task",
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=len(task_panels),
        fontsize=9,
        title_fontsize=10,
    )
    fig.tight_layout(rect=(0.0, 0.11, 1.0, 1.0), h_pad=1.2, w_pad=0.1)
    fig.savefig(TOOL_COMPARISON_PLOTS_DIR / "tool-use-by-tool-calls-sandboxed.pdf", bbox_inches="tight")
    plt.show()


def make_tool_call_plot_comparison(
    tool_plot_df: pd.DataFrame,
    task_names: list[str],
) -> None:
    plot_data = tool_plot_df[tool_plot_df["task_name"].isin(task_names)]
    task_panels = [task for task in task_names if task in set(plot_data["task_name"])]
    active_groups = [
        (key, display)
        for key, _labels, display in TOOL_GROUPS
        if display in set(plot_data["tool_display"])
    ]
    if not task_panels or not active_groups:
        raise ValueError("no matching math task/tool records to plot")

    summary = summarize_tool_calls(
        plot_data,
        ["source", "task_name", "N", "tool_key", "tool_display"],
    )
    cmap = plt.get_cmap("tab10")
    group_colors = {key: cmap(index) for index, (key, _display) in enumerate(active_groups)}
    fig, axes = plt.subplots(
        len(COMPARISON_SOURCES),
        len(task_panels),
        figsize=(2.4 * len(task_panels), 4.3),
        squeeze=False,
        sharex="col",
    )
    for row, source in enumerate(COMPARISON_SOURCES):
        for col, task_name in enumerate(task_panels):
            ax = axes[row][col]
            column_data = summary[summary["task_name"] == task_name]
            panel = column_data[column_data["source"] == source]
            for group_key, _display in active_groups:
                line = panel[panel["tool_key"] == group_key].sort_values("N")
                if line.empty or line["mean"].isna().all():
                    continue
                ax.errorbar(
                    line["N"],
                    line["mean"],
                    yerr=line["stderr"],
                    color=group_colors[group_key],
                    marker="o",
                    linewidth=1.6,
                    markersize=3.5,
                    capsize=2.0,
                )

            if row == 0:
                ax.set_title(task_name, fontsize=12)
                ax.tick_params(labelbottom=False)
            if col == 0:
                ax.set_ylabel(f"{source}\nMean Tool Calls")
            style_tool_comparison_axis(
                ax,
                sorted(column_data["N"].unique()),
                TASK_X_LABELS[task_name] if row == len(COMPARISON_SOURCES) - 1 else None,
            )

    tool_handles = [
        Line2D([0], [0], color=group_colors[key], marker="o", linewidth=1.8)
        for key, _display in active_groups
    ]
    fig.legend(
        tool_handles,
        [display for _key, display in active_groups],
        title="Tool Call Type",
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=len(active_groups),
        fontsize=9,
        title_fontsize=10,
    )
    fig.tight_layout(rect=(0.0, 0.11, 1.0, 1.0), h_pad=1.2, w_pad=0.1)
    fig.savefig(TOOL_COMPARISON_PLOTS_DIR / "tool-use-by-task-calls-sandboxed.pdf", bbox_inches="tight")
    plt.show()


COMPARISON_TOOL_NAMES = ["Analyze", "Read", "Search", "Edit", "Test"]
make_tool_call_plot_by_tool_column_comparison(
    tool_call_comparison_df,
    MATH_TASK_ORDER,
    COMPARISON_TOOL_NAMES,
)
make_tool_call_plot_comparison(tool_call_comparison_df, MATH_TASK_ORDER)